In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import requests
import tempfile
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from transformers import AutoTokenizer
from langchain_core.documents import Document


# Load variables from .env into environment
load_dotenv()

# Read the key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")
EMBED_MODEL = os.getenv("EMBED_MODEL")
COLLECTION_NAME = os.getenv("COLLECTION_NAME")

d:\Project\AI\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
def download_pdf(temp_content: bytes) -> str:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(temp_content)
        pdf_path = tmp.name
    return pdf_path

In [2]:
from pathlib import Path
current_dir = Path.cwd()
BASE_DIR = current_dir.parent.parent
root_folder = BASE_DIR / "dataset/policies/medishield_gold_plan.pdf"

root_folder

WindowsPath('d:/Project/AI/Project/dataset/policies/medishield_gold_plan.pdf')

In [ ]:
#download_pdf('d:/Project/AI/Project/dataset/policies/medishield_gold_plan.pdf')

In [3]:
converter = DocumentConverter()
dl_doc = converter.convert(root_folder).document

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1448.19it/s]
[INFO] 2026-10-16 20:21:25,377 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-16 20:21:25,573 [RapidOCR] download_file.py:60: File exists and is valid: D:\Project\AI\Project\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-10-16 20:21:25,577 [RapidOCR] main.py:63: Using D:\Project\AI\Project\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-10-16 20:21:28,459 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-16 20:21:28,525 [RapidOCR] download_file.py:60: File exists and is valid: D:\Project\AI\Project\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-16 20:21:28,527 [RapidOCR] main.py:63: Using D:\Project\AI\Project\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-16 20:21:28,894 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2

In [4]:
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

chunker = HybridChunker(
    tokenizer=tokenizer,   # aligns chunk size to the embedding model's token limit
    max_tokens=128,        # max tokens per chunk
    merge_peers=True,      # merge undersized sibling chunks under the same heading
)

chunk_iter = chunker.chunk(dl_doc=dl_doc)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (892 > 512). Running this sequence through the model will result in indexing errors


In [5]:
pdf_docs = []
for chunk in chunk_iter:
    pdf_docs.append(Document(
        page_content=chunker.serialize(chunk=chunk),
        metadata={"source": "policy_guide"}
    ))

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14884\1841285833.py:4: DeprecationWarning: Use contextualize() instead.
  page_content=chunker.serialize(chunk=chunk),


In [6]:
SPARSE_MODEL = os.getenv("SPARSE_MODEL")

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import FastEmbedSparse

# Dense embeddings — semantic understanding
dense_embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},      # change to "cuda" if you have a GPU
    encode_kwargs={"normalize_embeddings": True}
)
print("Dense embedding model:", EMBED_MODEL)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1615.71it/s]


Dense embedding model: sentence-transformers/all-MiniLM-L6-v2


In [8]:
# Explicitly pass a valid string model name
sparse_embeddings = FastEmbedSparse(
    model_name="Qdrant/bm25",  # Must be a string, not None
    batch_size=32
)
print("Sparse embedding model: Qdrant/BM25")

Sparse embedding model: Qdrant/BM25


In [9]:
import shutil, os, gc
from qdrant_client import QdrantClient

STORAGE_PATH = "/tmp/my_lang_vs"

# Make sure no client from this session is holding the lock
if "client" in globals():
    try:
        client.close()
    except Exception:
        pass
    del client
gc.collect()

# Now safe to delete
if os.path.exists(STORAGE_PATH):
    shutil.rmtree(STORAGE_PATH)
    print(f"Deleted {STORAGE_PATH}")



Deleted /tmp/my_lang_vs


In [10]:
import os

from qdrant_client.http.exceptions import UnexpectedResponse
from langchain_qdrant import QdrantVectorStore, RetrievalMode

STORAGE_PATH = "/tmp/my_lang_vs"

vectorstore = QdrantVectorStore.from_documents(
        documents=pdf_docs,
        embedding=dense_embeddings,
        sparse_embedding=sparse_embeddings,
        path=STORAGE_PATH,  # Pass path directly, not a client
        collection_name=COLLECTION_NAME,
        retrieval_mode=RetrievalMode.HYBRID,
    )
    

In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [19]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Create a retriever in HYBRID mode
hybrid_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}       # retrieve top-5 docs
)

# Prompt template
system_prompt = """You are a helpful Medishield Policy support assistant.
Answer the customer's question using ONLY the information provided in the context below.
If the answer is not in the context, say "I don't have that information."
Keep answers concise and friendly.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Build the chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
hybrid_rag_chain = create_retrieval_chain(hybrid_retriever, question_answer_chain)

print("✅ Hybrid RAG chain ready")

✅ Hybrid RAG chain ready


In [30]:
def ask_hybrid(question: str):
    return hybrid_rag_chain.invoke({"input": question})

procedure_code = "17000"

result = ask_hybrid(
    question=(
        f"CPT procedure code {procedure_code}: "
        f"Retrieve relevant policy clauses and return a final coverage determination "
        f"as JSON with keys: covered (bool), coverage_percentage (float), "
        f"policy_clause (str), exclusions (list[str])."
    )
)
print(result)

{'input': 'CPT procedure code 17000: Retrieve relevant policy clauses and return a final coverage determination as JSON with keys: covered (bool), coverage_percentage (float), policy_clause (str), exclusions (list[str]).', 'context': [Document(metadata={'source': 'policy_guide', '_id': '1e29a45ed4dc4d00b94214d3350e8e28', '_collection_name': 'policy_agent_collection'}, page_content='4.1 Excluded CPT Code Ranges (Machine-Readable)\nThe table below lists CPT code ranges explicitly excluded from the MediShield Gold Plan. Claims with codes in these ranges are automatically denied unless a qualifying rider is active. Codes not listed here are subject to medical necessity review per Section 5.\n15000-15999, Procedure Category = Integumentary / Skin Grafts (Cosmetic). 15000-15999, Exclusion Basis = Cosmetic S4(a)'), Document(metadata={'source': 'policy_guide', '_id': 'f9a9c0dc19d247a9aa95ad1181780e38', '_collection_name': 'policy_agent_collection'}, page_content='4.1 Excluded CPT Code Ranges (

In [34]:
print(result['answer'])

{
  "covered": false,
  "coverage_percentage": 0.0,
  "policy_clause": "4.1 Excluded CPT Code Ranges (Machine-Readable)",
  "exclusions": [
    "Cosmetic S4(a)"
  ]
}
